In [80]:
#the real dataset
from datasets import load_dataset

dataset = load_dataset('fhswf/german_handwriting')


In [81]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 10854
    })
})


In [82]:
print(dataset['train'][0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=3024x547 at 0x140819220>, 'text': '- Terminvorschlag bis'}


In [83]:
subset = dataset['train'].select(range(500))

In [84]:
from PIL import Image

subset = dataset['train'].select(range(500))

widths, heights = [], []
for row in subset:
    w, h = row['image'].size
    widths.append(w)
    heights.append(h)

print("width  min/max/avg:", min(widths), max(widths), sum(widths) / len(widths))
print("height min/max/avg:", min(heights), max(heights), sum(heights) / len(heights))

width  min/max/avg: 569 4080 2457.13
height min/max/avg: 35 547 91.762


In [85]:
vocab = set()
for row in subset:
    vocab.update(row['text'])

print(vocab)
print(len(vocab))

{'"', 'A', 'd', 'ä', '?', 'g', '1', 'F', 'ß', 'z', 'ö', 'm', 'I', 'U', 'M', 'v', 's', '-', '(', ';', '&', '4', 'Z', 'O', 'P', 'E', 'r', 'D', ',', ')', 'b', '+', 'Q', 'f', 'l', 'B', 'R', 'a', 'h', 'e', '2', 'u', '>', '/', 'S', 'V', 'q', 'H', 'N', 'i', ' ', '.', 'Ä', 'L', 'c', 'T', '|', 'k', '7', '9', 'w', 'y', 'J', 'G', 'Ü', 'K', '5', 'n', '0', 'ü', 't', 'C', 'W', '8', '3', 'o', 'p', ':', '=', 'x', 'j', '6', '\\'}
83


In [86]:
#need to get all the images in the dataset for a common height  - checking the median height
import statistics
print(statistics.median(heights))

83.0


In [87]:
#resizig
def resize_to_height(img, target_height = 64):
    orig_w, orig_h = img.size
    scale = target_height / orig_h
    new_w = int(orig_w * scale)
    return img.resize((new_w, target_height))

In [88]:
resized = resize_to_height(subset[0]['image']);print(resized.size)  # resized correctly

(353, 64)


In [89]:
#resizig
def resize_to_height(img, target_height = 64):
    orig_w, orig_h = img.size
    scale = target_height / orig_h
    new_w = int(orig_w * scale)
    return img.resize((new_w, target_height))

In [90]:
#creating dictionaries like lookup tables, 
sorted_vocab = sorted(vocab)
char2idx = {char: idx + 1 for idx, char in enumerate(sorted_vocab)}   # +1 leaves 0 free for blank , this maps characters to numbers
idx2char = {idx: char for char, idx in char2idx.items()} # this maps numbers to characters again

In [91]:
from torch.utils.data import Dataset
import torch 
from torchvision import transforms # ready made image processing steps

to_tensor = transforms.ToTensor() # reusable converter object

class HandwritingDataset(Dataset): #Dataset in this what makes this object is usable by DataLoader later
    #should hand in __len__ and also __getitem__

    def __init__(self,hf_dataset,char2idx,target_height=64):  # creating the constrcutor to hand it in
       self.data = hf_dataset
       self.char2idx = char2idx
       self.target_height = target_height

    def __len__(self):
        return len(self.data) # calling in the __len__required   , answers how many examples in total
        
    def __getitem__(self, idx):
        row = self.data[idx]    

    #take the image in that idx and make it standard to what i ceated before
        image = resize_to_height(row['image'],self.target_height)    
        image_tensor = to_tensor(image) # image is converted to a tensor

    #now the transcription should be converted to a tensor
        label = [self.char2idx[c] for c in row['text']]
        label_tensor = torch.tensor(label, dtype=torch.long) # should be wrapped in torch.tensor to mae it a tensor

        return image_tensor, label_tensor




In [92]:
ds = HandwritingDataset(dataset['train'], char2idx)
img_tensor, label_tensor = ds[0] # triggers __getitem__
print(img_tensor.shape, label_tensor)

torch.Size([3, 64, 353]) tensor([ 8,  1, 45, 55, 68, 63, 59, 64, 72, 65, 68, 69, 53, 58, 62, 51, 57,  1,
        52, 59, 69])


In [93]:
batch = [ds[0], ds[1], ds[2]]      # a tiny 3-example "batch", built by hand
images, labels = zip(*batch)        # now images actually exists

widths = [img.shape[2] for img in images]
max_width = max(widths)
print(widths, max_width)

[353, 366, 256] 366


In [102]:
# the image tensors have different widths and the label tensors have different number of
# characters hence not the same dimension

def collate_fn(batch):
    images, labels = zip(*batch)

    ## batch is a list dataloader builds automatically it calls ds[0, 2] lets say 16 times and collects the result into one list of pairs
    ## then should find the widest element in the batch because different sized elements cannot stack on each other

    widths = [img.shape[2] for img in images]  # img.shape on a tensor gives the 3 items that it has and [2] selects the width because its at the end
    max_width = max(widths)

    padded_images = []
    for img in images:
        pad_amount = max_width - img.shape[2]
        padded_img = torch.nn.functional.pad(img, (0, pad_amount))
        padded_images.append(padded_img)

    image_batch = torch.stack(padded_images)

    # doing the same to labels
    label_lengths = [len(label) for label in labels]
    max_label_len = max(label_lengths)

    padded_labels = []
    for label in labels:
        pad_amount = max_label_len - len(label)
        padded_label = torch.nn.functional.pad(label, (0, pad_amount), value=0)
        padded_labels.append(padded_label)

    label_batch = torch.stack(padded_labels)
    label_lengths = torch.tensor(label_lengths, dtype=torch.long)

    return image_batch, label_batch, label_lengths

In [95]:
# actually padding each image
padded_images = []
for img in images : 
    pad_amount = max_width - img.shape[2]
    padded_img = torch.nn.functional.pad(img, (0, pad_amount)) # add 0 pixels to the left side of the width and pad_amout to the right side of the width
    padded_images.append(padded_img) 


print([img.shape for img in padded_images])

[torch.Size([3, 64, 366]), torch.Size([3, 64, 366]), torch.Size([3, 64, 366])]


In [96]:
#turting the list of same- width tensors in to single batch tensor
image_batch = torch.stack(padded_images)
image_batch.shape
#new 3 infront is the batch size 

torch.Size([3, 3, 64, 366])

In [103]:
batch = [ds[0], ds[1], ds[2]]
image_batch, label_batch, label_lengths = collate_fn(batch)

print(image_batch.shape)
print(label_batch.shape)
print(label_lengths)

torch.Size([3, 3, 64, 366])
torch.Size([3, 21])
tensor([21, 19, 11])
